# Fullback Analysis System Documentation

## Overview
The Fullback Analysis System is a sophisticated Python-based tool designed to analyze and evaluate the performance of football (soccer) fullbacks using StatsBomb data. The system processes match events to generate comprehensive performance metrics across multiple dimensions of fullback play.

## Core Features
- Multi-league and multi-season analysis capability
- Comprehensive performance metrics calculation
- Normalized scoring system (4-10 scale)
- Confidence interval calculations
- Weighted averaging based on minutes played
- Duplicate player handling across competitions

## Key Performance Metrics

### 1. Overlap/Inversion Score
Measures a fullback's tendency to overlap or invert based on:
- Pass angles (negative angles indicate inversion, positive indicate overlap)
- Vertical movement patterns
- Weighted combination of angle score (60%) and movement score (40%)

### 2. Final Third Entries
Evaluates the fullback's efficiency in entering the final third through:
- Success rate of passes into the final third (70% weight)
- Frequency of final third entries (30% weight)
- Considers passes crossing the 70-yard line mark

### 3. Defensive Performance
Analyzes defensive capabilities through:
- Duel success rate (60% weight)
  - Includes tackles and recoveries
  - Calculated as won duels / total duels
- Pressure success rate (40% weight)
  - Measures successful pressure applications leading to ball recovery

### 4. Ball Progression
Evaluates the fullback's ability to advance the ball through:
- Progressive carries (40% weight)
  - Minimum 30 yards forward in own half
  - Minimum 15 yards forward in opposition half
- Progressive passes (60% weight)
  - Success rate of forward-advancing passes
  - Uses same distance thresholds as carries

### 5. Chance Creation
Measures attacking contribution through:
- Cross effectiveness (40% weight)
  - Completion rate
  - Dangerous cross positioning
- Shot/Goal Creation (40% weight)
  - Shot assist rate
  - Goal assist rate
- Box entries (20% weight)
  - Rate of successful passes into the penalty area

## Data Processing Pipeline

### 1. Match Data Collection
```python
matches = sb.matches(competition_id=competition_id, season_id=season_id)
```
- Retrieves match data from StatsBomb API
- Processes matches by competition and season

### 2. Event Processing
- Filters events for fullbacks (Right Back, Left Back positions)
- Calculates minutes played including added time
- Processes individual events for metric calculation

### 3. Metric Normalization
- Uses dynamic range (4-10 scale)
- Adjusts based on:
  - Sample size reliability
  - Minutes played
  - Performance distribution
- Includes confidence interval calculations

### 4. Data Aggregation
- Combines data across matches
- Handles duplicate players by keeping highest minutes played
- Calculates composite scores
- Generates final rankings

## Usage

### Basic Usage
```python
# Define leagues for analysis
leagues = [
    {"competition_id": 9, "season_ids": [281]},
    {"competition_id": 43, "season_ids": [106]},
    # Add more leagues as needed
]

# Run analysis
final_analysis = analyze_multiple_leagues(leagues)
```

### Output
- Generates CSV file: "combined_fullback_analysis.csv"
- Displays summary statistics including:
  - Total unique fullbacks analyzed
  - Average matches played
  - Average minutes played
  - Top 10 performers with detailed metrics

## Data Requirements
- Requires StatsBomb data access
- Minimum recommended data:
  - 90 minutes of play time per player
  - Multiple matches for reliable analysis
  - Complete event data including:
    - Pass locations and outcomes
    - Defensive actions
    - Carry information
    - Player positioning

## Performance Considerations
- Handles missing data gracefully
- Implements error handling for API calls
- Uses weighted averages for more accurate representation
- Adjusts reliability based on minutes played

## Limitations
- Dependent on StatsBomb data quality
- Requires minimum playing time for reliable analysis
- May not capture all nuanced aspects of fullback play
- Limited to available competition data

## Technical Dependencies
- pandas
- numpy
- statsbombpy
- warnings (for warning suppression)
- collections (defaultdict)

In [ ]:
import pandas as pd
import numpy as np
from statsbombpy import sb
from collections import defaultdict
import sys
import glob
import os
import warnings
warnings.simplefilter("ignore")

pd.set_option('display.max_rows', None)



pd.set_option('display.max_rows', None)

def calculate_overlap_inversion(events, fullback, team):
    """
    Analyze fullback's tendency to overlap or invert based on pass angles and vertical movement
    Returns a score between 0-1
    """
    fullback_events = events[events['player'] == fullback]
    
    # Get passes
    passes = fullback_events[fullback_events['type'] == 'Pass']
    
    if len(passes) == 0:
        return 0
    
    # Calculate average pass angle
    # Negative = inverting, Positive = overlapping
    pass_angles = passes['pass_angle'].mean() if 'pass_angle' in passes.columns else 0
    
    # Calculate vertical movement
    def calculate_vertical_distance(start_loc, end_loc):
        if not (isinstance(start_loc, list) and isinstance(end_loc, list)):
            return 0
        return end_loc[1] - start_loc[1]  # y-coordinate difference
    
    vertical_movements = passes.apply(
        lambda x: calculate_vertical_distance(x['location'], x['pass_end_location']), 
        axis=1
    ).mean()
    
    # Normalize scores
    angle_score = (pass_angles + 90) / 180  # Convert -90 to 90 range to 0-1
    movement_score = (vertical_movements + 40) / 80  # Normalize assuming max 40 units movement
    
    return (angle_score * 0.6) + (movement_score * 0.4)

def calculate_final_third_entries(events, fullback, team):
    """
    Analyze fullback's efficiency in entering the final third
    Returns a score between 0-1
    """
    fullback_events = events[events['player'] == fullback]
    
    def is_final_third_entry(start_loc, end_loc):
        if not (isinstance(start_loc, list) and isinstance(end_loc, list)):
            return False
        return start_loc[0] < 70 and end_loc[0] >= 70
    
    # Get passes into final third
    passes = fullback_events[fullback_events['type'] == 'Pass']
    final_third_entries = passes[
        passes.apply(lambda x: is_final_third_entry(x['location'], x['pass_end_location']), axis=1)
    ]
    
    if len(passes) == 0:
        return 0
    
    # Calculate success rate
    entry_success_rate = len(final_third_entries[final_third_entries['pass_outcome'].isna()]) / max(len(final_third_entries), 1)
    entry_frequency = len(final_third_entries) / len(passes)
    
    return (entry_success_rate * 0.7) + (entry_frequency * 0.3)

def calculate_defensive_performance(events, fullback, team):
    """
    Calculate defensive metrics including duels and pressure success
    Returns a score between 0-1
    """
    fullback_events = events[events['player'] == fullback]
    
    # Get defensive duels
    duels = fullback_events[
        (fullback_events['type'] == 'Duel') &
        (fullback_events['duel_type'].isin(['Tackle', 'Recovery']))
    ]
    
    if len(duels) == 0:
        return 0
    
    # Calculate duel success rate
    duel_success_rate = len(duels[duels['duel_outcome'] == 'Won']) / len(duels)
    
    # Calculate pressure success
    pressures = fullback_events[fullback_events['type'] == 'Pressure']
    pressure_success = 0
    if len(pressures) > 0:
        def pressure_successful(events, pressure_idx):
            if pressure_idx + 1 >= len(events):
                return False
            next_event = events.iloc[pressure_idx + 1]
            return next_event['team'] == team
            
        successful_pressures = sum(
            pressure_successful(events, i)
            for i in pressures.index
        )
        pressure_success = successful_pressures / len(pressures)
    
    return (duel_success_rate * 0.6) + (pressure_success * 0.4)

def calculate_ball_progression(events, fullback, team):
    """
    Calculate ball progression metrics including progressive carries and passes
    Returns a score between 0-1
    """
    fullback_events = events[events['player'] == fullback]
    
    def is_progressive(start_loc, end_loc):
        if not (isinstance(start_loc, list) and isinstance(end_loc, list)):
            return False
        dist_threshold = 30 if start_loc[0] < 50 else 15  # Different thresholds for own/opposition half
        return end_loc[0] - start_loc[0] >= dist_threshold
    
    # Progressive carries
    carries = fullback_events[fullback_events['type'] == 'Carry']
    progressive_carries = carries[
        carries.apply(lambda x: is_progressive(x['location'], x['carry_end_location']), axis=1)
    ]
    carry_score = len(progressive_carries) / max(len(carries), 1)
    
    # Progressive passes
    passes = fullback_events[fullback_events['type'] == 'Pass']
    progressive_passes = passes[
        passes.apply(lambda x: is_progressive(x['location'], x['pass_end_location']), axis=1)
    ]
    
    if len(passes) == 0:
        return 0
        
    pass_success = len(progressive_passes[progressive_passes['pass_outcome'].isna()]) / max(len(progressive_passes), 1)
    
    return (carry_score * 0.4) + (pass_success * 0.6)

def calculate_chance_creation(events, fullback, team):
    """
    Calculate chance creation metrics including crossing, shot creation, and attacking contribution
    Returns a score between 0-1
    """
    fullback_events = events[events['player'] == fullback]
    
    # Get all attacking actions
    attacking_actions = fullback_events[fullback_events['type'].isin(['Pass', 'Cross', 'Shot Assist', 'Goal Assist'])]
    if len(attacking_actions) == 0:
        return 0
    
    # 1. Analyze crosses with more detail
    crosses = attacking_actions[
        (attacking_actions['type'] == 'Pass') &
        (attacking_actions['pass_type'].isin(['Cross', 'Low Cross', 'High Pass']))
    ]
    
    cross_score = 0
    if len(crosses) > 0:
        # Successful crosses (completed or created chance)
        successful_crosses = crosses[
            crosses['pass_outcome'].isna() |
            crosses['pass_shot_assist'] == True |
            crosses['pass_goal_assist'] == True
        ]
        cross_completion = len(successful_crosses) / len(crosses)
        
        # Weight crosses by location (final third weighted higher)
        def is_final_third(location):
            return isinstance(location, list) and location[0] >= 70
            
        dangerous_crosses = len(crosses[crosses['location'].apply(is_final_third)]) / len(crosses)
        cross_score = (cross_completion * 0.6) + (dangerous_crosses * 0.4)
    
    # 2. Shot/Goal Creation
    shot_assists = fullback_events[fullback_events['pass_shot_assist'] == True]
    goal_assists = fullback_events[fullback_events['pass_goal_assist'] == True]
    
    total_passes = len(fullback_events[fullback_events['type'] == 'Pass'])
    if total_passes == 0:
        return 0
    
    # Calculate creation rates
    shot_creation_rate = len(shot_assists) / total_passes
    goal_creation_rate = len(goal_assists) / total_passes
    
    # 3. Progressive Passes into Box
    def is_box_entry(start_loc, end_loc):
        if not (isinstance(start_loc, list) and isinstance(end_loc, list)):
            return False
        return end_loc[0] >= 75 and 15 <= end_loc[1] <= 65
    
    box_entries = attacking_actions[
        attacking_actions.apply(lambda x: 'pass_end_location' in x and 
                              is_box_entry(x['location'], x['pass_end_location']), axis=1)
    ]
    box_entry_rate = len(box_entries) / total_passes
    
    # Weight the components
    creation_score = (
        (cross_score * 0.4) +
        (shot_creation_rate * 0.2) +
        (goal_creation_rate * 0.2) +
        (box_entry_rate * 0.2)
    )
    
    # Normalize to ensure 0-1 range
    return min(max(creation_score, 0), 1)

def normalize_metric(series, min_val=4, max_val=10, minutes_played=None):
    """
    Improved metric normalization with better handling of edge cases
    """
    if len(series) == 0:
        return pd.Series([]), 0
    
    if minutes_played is not None:
        # Increase minimum minutes threshold
        min_minutes = 90  # One full match minimum
        reliability_factor = np.minimum(minutes_played / (4 * 90), 1)  # Scale up to 4 matches
        weights = reliability_factor
        series = series * weights
    
    if series.std() < 1e-10:
        return pd.Series([6.5] * len(series)), 0
    
    # Use more robust bounds based on sample size
    if len(series) < 10:
        # For small samples, use min/max with some padding
        lower_bound = series.min() - (series.std() * 0.5)
        upper_bound = series.max() + (series.std() * 0.5)
    else:
        # For larger samples, use percentiles
        lower_bound = series.quantile(0.05)  # 5th percentile
        upper_bound = series.quantile(0.95)  # 95th percentile
    
    # Normalize and scale
    normalized = (series - lower_bound) / (upper_bound - lower_bound)
    normalized = np.clip(normalized, 0, 1)
    
    # Dynamic range based on sample reliability
    if minutes_played is not None:
        # Compress range for less reliable samples
        min_val_adj = min_val + (1 - reliability_factor.mean()) * 2
        max_val_adj = max_val - (1 - reliability_factor.mean()) * 2
    else:
        min_val_adj = min_val
        max_val_adj = max_val
    
    scaled = normalized * (max_val_adj - min_val_adj) + min_val_adj
    
    # Add confidence intervals
    std_err = series.std() / np.sqrt(len(series))
    conf_interval = 1.96 * std_err  # 95% confidence interval
    
    return scaled.round(1), conf_interval


def analyze_fullback_performance(competition_id, season_id):
    """
    Analyze fullback performance across multiple metrics, aggregated across all matches
    Returns a DataFrame with normalized scores and confidence intervals
    """
    matches = sb.matches(competition_id=competition_id, season_id=season_id)
    all_fb_stats = []
    
    for _, match in matches.iterrows():
        try:
            events = sb.events(match_id=match['match_id'])
            fullbacks = events[
                events['position'].isin(['Right Back', 'Left Back'])
            ]['player'].unique()
            
            for fb in fullbacks:
                fb_events = events[events['player'] == fb]
                if len(fb_events) == 0:
                    continue
                
                team = fb_events['team'].iloc[0]
                
                # Calculate minutes played including added time
                start_minute = fb_events['minute'].min()
                end_minute = fb_events['minute'].max()
                added_time = fb_events['second'].max() / 60 if end_minute >= 90 else 0
                minutes = end_minute - start_minute + added_time
                
                # Calculate all metrics
                metrics = {
                    'fullback': fb,
                    'team': team,
                    'match_id': match['match_id'],
                    'minutes_played': minutes,
                    'overlap_inversion': calculate_overlap_inversion(events, fb, team),
                    'final_third_entries': calculate_final_third_entries(events, fb, team),
                    'defensive_performance': calculate_defensive_performance(events, fb, team),
                    'ball_progression': calculate_ball_progression(events, fb, team),
                    'chance_creation': calculate_chance_creation(events, fb, team)
                }
                
                all_fb_stats.append(metrics)
                
        except Exception as e:
            print(f"Error processing match {match['match_id']}: {str(e)}")
            continue
    
    # Create DataFrame
    df = pd.DataFrame(all_fb_stats)
    if len(df) == 0:
        return pd.DataFrame()
    
    # Group by fullback and calculate weighted averages
    metrics = ['overlap_inversion', 'final_third_entries', 'defensive_performance', 
              'ball_progression', 'chance_creation']
    
    def weighted_avg(x):
        return np.average(x, weights=df.loc[x.index, 'minutes_played'])
    
    grouped_df = df.groupby(['fullback', 'team']).agg({
        'minutes_played': 'sum',
        **{metric: weighted_avg for metric in metrics}
    }).reset_index()
    
    # Add matches played and full matches equivalent
    matches_played = df.groupby('fullback')['match_id'].nunique()
    grouped_df['matches_played'] = grouped_df['fullback'].map(matches_played)
    grouped_df['full_match_equivalent'] = (grouped_df['minutes_played'] / 90).round(1)
    
    # Normalize all metrics and add confidence intervals
    normalized_metrics = {}
    confidence_intervals = {}
    
    for metric in metrics:
        normalized, conf_interval = normalize_metric(
            grouped_df[metric], 
            minutes_played=grouped_df['minutes_played']
        )
        normalized_metrics[metric] = normalized
        confidence_intervals[f'{metric}_ci'] = conf_interval
    
    # Add normalized metrics and confidence intervals to DataFrame
    for metric in metrics:
        grouped_df[metric] = normalized_metrics[metric]
        grouped_df[f'{metric}_ci'] = confidence_intervals[f'{metric}_ci']
    
    # Calculate composite score
    grouped_df['composite_score'] = grouped_df[metrics].mean(axis=1)
    
    # Sort by composite score
    grouped_df = grouped_df.sort_values('composite_score', ascending=False)
    
    # Round specific columns
    round_cols = metrics + [m + '_ci' for m in metrics] + ['composite_score', 'minutes_played']
    grouped_df[round_cols] = grouped_df[round_cols].round(2)
    
    # Set fullback as index
    return grouped_df.set_index('fullback')




def analyze_multiple_leagues(leagues):
    """
    Analyze fullback performance across multiple leagues and seasons
    Handles duplicate players by keeping only their stats from the competition 
    where they played the most minutes
    """
    all_analyses = []
    
    for league in leagues:
        competition_id = league["competition_id"]
        for season_id in league["season_ids"]:
            try:
                print(f"Processing competition {competition_id}, season {season_id}...")
                analysis = analyze_fullback_performance(competition_id, season_id)
                if not analysis.empty:
                    # Add competition and season info
                    analysis['competition_id'] = competition_id
                    analysis['season_id'] = season_id
                    all_analyses.append(analysis)
            except Exception as e:
                print(f"Error processing competition {competition_id}, season {season_id}: {str(e)}")
                continue
    
    if not all_analyses:
        print("No data found for any league/season combination")
        return pd.DataFrame()
    
    # Combine all analyses
    combined_df = pd.concat(all_analyses, axis=0)
    
    # Handle duplicate players by keeping only the entry with most minutes played
    final_df = combined_df.sort_values('minutes_played', ascending=False)
    final_df = final_df.groupby(final_df.index).first().reset_index()
    
    # Sort by composite score
    final_df = final_df.sort_values('composite_score', ascending=False)
    
    # Save to CSV
    output_filename = "combined_fullback_analysis.csv"
    final_df.to_csv(output_filename)
    print(f"\nResults saved to {output_filename}")
    
    return final_df

if __name__ == "__main__":
    leagues = [
        {"competition_id": 9, "season_ids": [281]},
        {"competition_id": 43, "season_ids": [106]},
        {"competition_id": 11, "season_ids": [90, 42, 4]},
        {"competition_id": 7, "season_ids": [235, 108]},
        {"competition_id": 2, "season_ids": [44]},
        {"competition_id": 12, "season_ids": [27]},
        {"competition_id": 55, "season_ids": [282]},
    ]
    
    print("Starting multi-league fullback analysis...")
    final_analysis = analyze_multiple_leagues(leagues)
    
    if not final_analysis.empty:
        # Display summary statistics
        print("\nAnalysis Summary:")
        print(f"Total unique fullbacks analyzed: {len(final_analysis)}")
        print(f"Average matches played: {final_analysis['matches_played'].mean():.1f}")
        print(f"Average minutes played: {final_analysis['minutes_played'].mean():.1f}")
        
        # Display top 10 performers
        print("\nTop 10 Fullbacks by Composite Score:")
        print("-" * 100)
        display_columns = [
            'team', 
            'competition_id',
            'season_id',
            'matches_played',
            'minutes_played',
            'composite_score',
            'overlap_inversion',
            'final_third_entries',
            'defensive_performance',
            'ball_progression',
            'chance_creation'
        ]
        print(final_analysis[display_columns].head(10))
    else:
        print("No data was processed successfully.")